In [1]:
4+4

8

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage,SystemMessage,HumanMessage
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_classic.chains import create_history_aware_retriever
load_dotenv()

llm=ChatOpenAI(model='gpt-3.5-turbo')
loader=WebBaseLoader('https://www.flipkart.com/apple-iphone-15-black-128-gb/p/itm3234701b0863a')
docs=loader.load()
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=75)
split=splitter.split_documents(docs)
vector=Chroma.from_documents(documents=split,embedding=OpenAIEmbeddings())
retriever=vector.as_retriever()

system_data='you are an helpful question answering system. Answer the query from the provided context.' \
'If you dont know the answer directly say thank you keep the answer concise {context}'

prompt=ChatPromptTemplate.from_messages([('system',system_data),('human','{input}')])
que_ans_chain=create_stuff_documents_chain(llm,prompt)
rag_chain=create_retrieval_chain(retriever,que_ans_chain)
'''response=rag_chain.invoke({'input':'what is price of the item'})
response['answer']'''
context_system='Given a chat history and latest user question which might reference context in chat history the user and dont answer with looking in to chat history'
context_prompt=ChatPromptTemplate.from_messages([('system','Reframe the query as a standalone query based on the past context or chat history'),
                                                 MessagesPlaceholder('chat_history'),('human','{input}')])

qa_prompt=ChatPromptTemplate.from_messages([('system', context_system),MessagesPlaceholder('chat_history'),('human','{input} \n\n Context: \n {context}')])
history_aware_retriever=create_history_aware_retriever(llm,retriever,context_prompt)
que_ans_chain=create_stuff_documents_chain(llm,qa_prompt)
rag_chain=create_retrieval_chain(history_aware_retriever,que_ans_chain)
chat_history=[]
question='what is the price of Iphone15'

response=rag_chain.invoke({'input':question,'chat_history':chat_history})
response['answer']
chat_history.extend([HumanMessage(content=question), AIMessage(content=response['answer'])])

question1='what query i asked you before'
response2=rag_chain.invoke({'input':question1,'chat_history':chat_history})

response2['answer']
